# 分类任务与损失函数：从前向传播到反向传播

这个小专题只使用 NumPy，完整实现一个三分类网络。我们会把前面学过的内容串起来：

`输入 → 线性层 → ReLU → logits → softmax → 交叉熵 → 反向传播 → 更新权重`

重点不是记住一段代码，而是看清楚每个张量的形状、每个梯度从哪里来，以及损失函数如何把分类错误传回网络。

## 学完后应该能回答什么？

- 什么问题属于分类，什么问题属于回归？
- 为什么多分类通常使用 `softmax + cross entropy`？
- 为什么交叉熵的梯度会变成 `probabilities - labels`？
- 一个 batch 中，`dW`、`db` 的形状为什么是这样？
- 如何用数值梯度检查手写的反向传播是否正确？

## 分类的适用范围

分类的目标不是预测一个连续数，而是从若干类别中判断样本属于哪一类。

| 问题类型 | 输出形式 | 常用组合 | 例子 |
| --- | --- | --- | --- |
| 二分类 | 一个概率 | sigmoid + binary cross entropy | 垃圾邮件 / 正常邮件 |
| 多分类 | 所有类别概率之和为 1 | softmax + categorical cross entropy | 猫 / 狗 / 鸟，三选一 |
| 多标签分类 | 每个类别独立判断 | 多个 sigmoid + binary cross entropy | 一张图片同时有猫和汽车 |

本 notebook 实现的是**互斥多分类**：每个样本只能属于一个类别，因此使用 softmax。多标签任务不能使用 softmax，因为多个类别可以同时为真。

## 先固定形状

假设：

- batch 中有 `N` 个样本；
- 每个样本有 `D` 个输入特征；
- 隐藏层有 `H` 个神经元；
- 一共有 `C` 个类别。

那么：

```text
X       : (N, D)
W1      : (D, H)
b1      : (H,)
hidden  : (N, H)
W2      : (H, C)
b2      : (C,)
logits  : (N, C)
labels  : (N, C)   # one-hot
```

广播规则让 `(N, H) + (H,)` 和 `(N, C) + (C,)` 可以对 batch 中每个样本加上同一个偏置。

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n_per_class = 60
centers = np.array([[-2.0, -1.0], [2.0, -1.0], [0.0, 2.0]])

X = np.vstack([
    center + 0.6 * rng.normal(size=(n_per_class, 2))
    for center in centers
])
y = np.repeat(np.arange(3), n_per_class)
labels = np.eye(3)[y]  # 整数标签 → one-hot 标签

print('X.shape =', X.shape)
print('y.shape =', y.shape)
print('labels.shape =', labels.shape)
print('前 5 个整数标签 =', y[:5])
print('前 5 个 one-hot 标签 =\n', labels[:5])


## softmax 和交叉熵

logits 是还没有归一化的类别分数。softmax 把它们转换为概率：

$$p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

交叉熵在 one-hot 标签下只关心真实类别的概率：

$$L = -\log(p_{\text{true}})$$

下面的 `axis=-1` 表示沿着类别维度计算；对一个 batch 来说，先得到每个样本的 loss，再求平均。

In [ ]:
def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values, axis=-1, keepdims=True)


def cross_entropy(probabilities, one_hot_labels):
    probabilities = np.asarray(probabilities, dtype=float)
    one_hot_labels = np.asarray(one_hot_labels, dtype=float)
    safe_probabilities = np.clip(probabilities, 1e-12, 1.0)
    per_sample_loss = -np.sum(
        one_hot_labels * np.log(safe_probabilities),
        axis=-1
    )
    return np.mean(per_sample_loss)


test_logits = np.array([[2.0, 1.0, 0.0], [0.0, 2.0, 1.0]])
test_labels = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
test_probabilities = softmax(test_logits)

print('probabilities =\n', test_probabilities)
print('每行的概率和 =', test_probabilities.sum(axis=-1))
print('batch loss =', cross_entropy(test_probabilities, test_labels))


## 两层网络的前向传播

网络结构是：

```text
X --线性层--> z1 --ReLU--> hidden --线性层--> logits --softmax--> probabilities
```

注意：训练时通常保留 `logits` 和 `hidden`，因为反向传播需要它们。这个“前向计算时保存中间结果”的对象，就是 cache。

In [ ]:
def relu(x):
    return np.maximum(x, 0.0)


def forward(X, W1, b1, W2, b2):
    z1 = X @ W1 + b1
    hidden = relu(z1)
    logits = hidden @ W2 + b2
    probabilities = softmax(logits)
    cache = (X, z1, hidden, probabilities)
    return probabilities, cache


input_size = X.shape[1]
hidden_size = 8
num_classes = labels.shape[1]

W1 = rng.normal(0.0, 0.3, size=(input_size, hidden_size))
b1 = np.zeros(hidden_size)
W2 = rng.normal(0.0, 0.3, size=(hidden_size, num_classes))
b2 = np.zeros(num_classes)

probabilities, cache = forward(X, W1, b1, W2, b2)
print('probabilities.shape =', probabilities.shape)
print('初始 loss =', cross_entropy(probabilities, labels))


## 反向传播：从损失往回走

对 softmax + 交叉熵，logits 的梯度可以直接写成：

$$d\_logits = \frac{probabilities - labels}{N}$$

除以 `N` 是因为我们对 batch 的样本损失取了平均。然后依次使用矩阵乘法的局部导数：

```text
dW2 = hidden.T @ dlogits
db2 = sum(dlogits, axis=0)
dhidden = dlogits @ W2.T
dz1 = dhidden * (z1 > 0)       # ReLU 的导数
dW1 = X.T @ dz1
db1 = sum(dz1, axis=0)
```

这里的每一步都遵循同一条原则：**上游梯度 × 当前运算的局部导数**。

In [ ]:
def backward(cache, labels, W2):
    X, z1, hidden, probabilities = cache
    batch_size = X.shape[0]

    dlogits = (probabilities - labels) / batch_size
    dW2 = hidden.T @ dlogits
    db2 = np.sum(dlogits, axis=0)

    dhidden = dlogits @ W2.T
    dz1 = dhidden * (z1 > 0)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0)

    return dW1, db1, dW2, db2


dW1, db1, dW2, db2 = backward(cache, labels, W2)
print('dW1.shape =', dW1.shape)
print('db1.shape =', db1.shape)
print('dW2.shape =', dW2.shape)
print('db2.shape =', db2.shape)


## 用梯度下降训练

每次训练的顺序固定为：

1. 用当前权重前向传播；
2. 计算交叉熵；
3. 反向传播得到所有梯度；
4. 用 `参数 -= learning_rate * 梯度` 更新参数。

这里使用整批数据，之后可以再改成 mini-batch。

In [ ]:
learning_rate = 0.15
steps = 1000

for step in range(steps + 1):
    probabilities, cache = forward(X, W1, b1, W2, b2)
    loss = cross_entropy(probabilities, labels)
    dW1, db1, dW2, db2 = backward(cache, labels, W2)

    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

    if step % 200 == 0:
        predictions = np.argmax(probabilities, axis=1)
        accuracy = np.mean(predictions == y)
        print(f'step={step:4d}, loss={loss:.4f}, accuracy={accuracy:.3f}')


## 数值梯度检查

手写反向传播时，最容易出现的是符号、转置或维度错误。数值梯度检查提供一个独立的验证方法。

对某个参数 `w`，用中心差分近似导数：

$$\frac{\partial L}{\partial w} \approx \frac{L(w+\epsilon)-L(w-\epsilon)}{2\epsilon}$$

如果手写梯度和数值梯度接近，说明反向传播大概率正确。梯度检查应该在小数据、随机参数上进行，而不是训练很久之后进行。

In [ ]:
# 在当前参数上检查 W1[0, 0]
probabilities, cache = forward(X, W1, b1, W2, b2)
analytic_grads = backward(cache, labels, W2)
analytic = analytic_grads[0][0, 0]

epsilon = 1e-5
original_value = W1[0, 0]

W1[0, 0] = original_value + epsilon
loss_plus = cross_entropy(forward(X, W1, b1, W2, b2)[0], labels)

W1[0, 0] = original_value - epsilon
loss_minus = cross_entropy(forward(X, W1, b1, W2, b2)[0], labels)

W1[0, 0] = original_value
numerical = (loss_plus - loss_minus) / (2 * epsilon)
relative_error = abs(analytic - numerical) / max(1e-12, abs(analytic) + abs(numerical))

print('analytic gradient =', analytic)
print('numerical gradient =', numerical)
print('relative error =', relative_error)


## 小结与练习

这次实现中，损失函数并不直接修改参数；它先产生梯度，优化器再负责更新参数：

```text
loss → dlogits → dW2/db2 → dhidden → dz1 → dW1/db1
```

建议你自己改动下面三个地方并观察结果：

- 把 `learning_rate` 改成 `0.01` 或 `1.0`；
- 把隐藏层大小从 `8` 改成 `2`；
- 把 `dlogits = (probabilities - labels) / batch_size` 临时改错，观察梯度检查如何报警。

记忆句：**分类损失评价概率，反向传播传递方向，梯度下降修改参数。**